# 🖼️ CBIS-DDSM Mini Dataset - Download & Preparation

## Mục tiêu
- Download mini CBIS-DDSM dataset (~5-10GB thay vì 163GB full)
- Convert DICOM to PNG images
- Dataset organization (Train/Val/Test splits)
- Image preprocessing pipelines
- Data augmentation setup
- Statistics về dataset

## 📊 CBIS-DDSM Dataset Overview

**CBIS-DDSM (Curated Breast Imaging Subset of DDSM)**
- Mammography images (X-ray of breast)
- Binary classification: Benign vs Malignant
- Full dataset: ~10,000 images, 163GB
- **Mini version**: Select subset for research purposes

**Format**: DICOM → Convert to PNG
**Modalities**: Mass, Calcification
**Views**: CC, MLO

## ⚠️ Storage Constraint

User cannot accommodate 163GB → Download strategic subset:
- 🎯 Target: 5-10GB total
- Sample from all classes proportionally
- Maintain train/val/test distribution

In [1]:
# Import libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Image processing
import cv2
from PIL import Image

# Medical imaging (DICOM)
try:
    import pydicom
    PYDICOM_AVAILABLE = True
except ImportError:
    PYDICOM_AVAILABLE = False
    print("⚠️ pydicom not installed - will need to install for DICOM processing")

# Sklearn
from sklearn.model_selection import train_test_split

# Custom imports
from src.utils.config import *
from src.utils.helpers import *

# Set random seed
set_seed(RANDOM_STATE)

print("✅ Libraries imported successfully")
print(f"pydicom available: {PYDICOM_AVAILABLE}")

🎲 Random seed set to: 42
✅ Libraries imported successfully
pydicom available: True


## 1. Setup Directories

In [2]:
# Create directories for CBIS-DDSM data
CBIS_DATA_DIR = PROJECT_ROOT / 'data' / 'cbis_ddsm'
CBIS_RAW_DIR = CBIS_DATA_DIR / 'raw'
CBIS_PROCESSED_DIR = CBIS_DATA_DIR / 'processed'
CBIS_IMAGES_DIR = CBIS_PROCESSED_DIR / 'images'

# Create subdirectories
for split in ['train', 'val', 'test']:
    for label in ['benign', 'malignant']:
        (CBIS_IMAGES_DIR / split / label).mkdir(parents=True, exist_ok=True)

print("✅ Directory structure created:")
print(f"  📁 {CBIS_DATA_DIR}")
print(f"    📁 raw/")
print(f"    📁 processed/")
print(f"      📁 images/")
print(f"        📁 train/ (benign, malignant)")
print(f"        📁 val/ (benign, malignant)")
print(f"        📁 test/ (benign, malignant)")

✅ Directory structure created:
  📁 /Users/GiangNguyenHuy/Documents/breast-cancer-ai/notebooks/../src/data/cbis_ddsm
    📁 raw/
    📁 processed/
      📁 images/
        📁 train/ (benign, malignant)
        📁 val/ (benign, malignant)
        📁 test/ (benign, malignant)


## 2. Download Strategy

### Option A: Kaggle API (Recommended)
**Dataset**: [CBIS-DDSM on Kaggle](https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset)

In [3]:
print_section_header("DOWNLOAD STRATEGY")

print("""
📥 DOWNLOAD CBIS-DDSM MINI DATASET

🎯 MINI SUBSET STRATEGY (5-10GB instead of 163GB):

Option 1: Kaggle Preprocessed Version
- Already converted to PNG
- Pre-organized directory structure
- Smaller file sizes
- URL: https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset

Option 2: TCIA Original (Full)
- Download full CBIS-DDSM from TCIA
- DICOM format (need conversion)
- Very large (163GB)
- Not recommended for this project

Option 3: Manual Subset Selection
- Download via Kaggle API with sample limit
- Select representative samples

📋 RECOMMENDED APPROACH:

1. Install Kaggle API:
   ```bash
   pip install kaggle
   ```

2. Setup Kaggle credentials:
   - Go to https://www.kaggle.com/settings
   - Create API token (downloads kaggle.json)
   - Place in ~/.kaggle/kaggle.json
   - chmod 600 ~/.kaggle/kaggle.json

3. Download dataset:
   ```bash
   cd data/cbis_ddsm/raw/
   kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset
   unzip cbis-ddsm-breast-cancer-image-dataset.zip
   ```

4. For MINI version - sample randomly:
   - Take 20-30% of each class
   - Target: ~2000-3000 images total (~5-10GB)

⚠️ NOTE: Due to storage constraints, this notebook provides:
   - Download instructions (manual)
   - Processing pipeline (automatic once data available)
   - Sample dataset statistics
   - Preprocessing functions ready to use

If you have limited storage, consider:
- Downloading only Mass OR Calcification (not both)
- Using lower resolution images
- Processing in batches
""")


                               DOWNLOAD STRATEGY                                


📥 DOWNLOAD CBIS-DDSM MINI DATASET

🎯 MINI SUBSET STRATEGY (5-10GB instead of 163GB):

Option 1: Kaggle Preprocessed Version
- Already converted to PNG
- Pre-organized directory structure
- Smaller file sizes
- URL: https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset

Option 2: TCIA Original (Full)
- Download full CBIS-DDSM from TCIA
- DICOM format (need conversion)
- Very large (163GB)
- Not recommended for this project

Option 3: Manual Subset Selection
- Download via Kaggle API with sample limit
- Select representative samples

📋 RECOMMENDED APPROACH:

1. Install Kaggle API:
   ```bash
   pip install kaggle
   ```

2. Setup Kaggle credentials:
   - Go to https://www.kaggle.com/settings
   - Create API token (downloads kaggle.json)
   - Place in ~/.kaggle/kaggle.json
   - chmod 600 ~/.kaggle/kaggle.json

3. Download dataset:
   ```bash
   cd data/cbis_ddsm/raw/
   kaggle datas

## 2b. Automated Download (Option - Run if you want to download)

### **🤖 Tự động download và tạo mini dataset**

**Workflow:**
1. Check Kaggle API installed
2. Download từ Kaggle (~10-15GB, mất 30-90 phút tùy internet)
3. Extract images
4. **Sample mini version** (chỉ lấy 20-30% → ~3-5GB)
5. Organize train/val/test splits

⚠️ **Chú ý:**
- Cần ~15-20GB free space tạm thời
- Sau khi sample mini, xóa full dataset → chỉ còn ~5GB
- Chi có chạy cell này **1 lần**

In [4]:
import subprocess
import shutil
import zipfile
from tqdm import tqdm

def check_kaggle_installed():
    """Check if kaggle CLI is installed"""
    try:
        result = subprocess.run(['kaggle', '--version'], 
                              capture_output=True, text=True)
        return True
    except FileNotFoundError:
        return False

def setup_kaggle_automated():
    """
    Automated Kaggle setup and CBIS-DDSM mini download
    
    ⚠️ RUN THIS CELL CHỈ 1 LẦN!
    """
    print_section_header("AUTOMATED KAGGLE DOWNLOAD")
    
    # Step 1: Check Kaggle installed
    print("\n1️⃣ Checking Kaggle CLI...")
    if not check_kaggle_installed():
        print("⚠️ Kaggle not installed. Installing...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'kaggle'])
        print("✅ Kaggle installed")
    else:
        print("✅ Kaggle CLI already installed")
    
    # Step 2: Check credentials
    print("\n2️⃣ Checking Kaggle credentials...")
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    
    if not kaggle_json.exists():
        print("""
⚠️ Kaggle credentials NOT found!

MANUAL SETUP REQUIRED:
1. Go to: https://www.kaggle.com/settings
2. Scroll to "API" section
3. Click "Create New API Token"
4. File 'kaggle.json' will download
5. Move it to: ~/.kaggle/kaggle.json
6. Run: chmod 600 ~/.kaggle/kaggle.json

Then re-run this cell.
""")
        return False
    
    print("✅ Kaggle credentials found")
    
    # Step 3: Download dataset
    print(f"\n3️⃣ Downloading CBIS-DDSM from Kaggle...")
    print("   ⏳ This will take 30-90 minutes depending on internet speed")
    print(f"   📦 Downloading to: {CBIS_RAW_DIR}")
    
    CBIS_RAW_DIR.mkdir(parents=True, exist_ok=True)
    
    dataset_name = "awsaf49/cbis-ddsm-breast-cancer-image-dataset"
    
    try:
        # Download
        subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', dataset_name, '-p', str(CBIS_RAW_DIR)],
            check=True
        )
        print("✅ Download complete!")
        
        # Step 4: Extract
        print("\n4️⃣ Extracting files...")
        zip_file = CBIS_RAW_DIR / 'cbis-ddsm-breast-cancer-image-dataset.zip'
        
        if zip_file.exists():
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(CBIS_RAW_DIR)
            print("✅ Extraction complete")
            
            # Remove zip to save space
            zip_file.unlink()
            print("✅ Removed zip file to save space")
        
        # Step 5: Find image directories
        print("\n5️⃣ Locating images...")
        
        # Kaggle dataset usually has structure:
        # raw/
        #   jpeg/
        #     train/ or similar
        #       benign/
        #       malignant/
        
        # Find actual image directory structure
        image_dirs = list(CBIS_RAW_DIR.rglob('*.png')) + list(CBIS_RAW_DIR.rglob('*.jpg'))
        
        if len(image_dirs) > 0:
            print(f"✅ Found {len(image_dirs)} images")
            
            # Step 6: Create MINI dataset
            print("\n6️⃣ Creating MINI dataset (sampling 25%)...")
            print("   This reduces size from ~15GB → ~4GB")
            
            # You need to manually find the source directory structure
            # and update this path based on actual Kaggle dataset structure
            print("""
⚠️ MANUAL STEP NEEDED:

After extraction, check the directory structure:
1. Navigate to data/cbis_ddsm/raw/
2. Find where the actual images are (benign/malignant folders)
3. Update the source_dir path below
4. Then run the sampling code

Example structures to look for:
- raw/jpeg/train/{benign,malignant}/
- raw/images/{benign,malignant}/
- raw/Mass-Training/...

Once found, use create_mini_dataset() function with correct path.
""")
            
            return True
        else:
            print("⚠️ No images found after extraction")
            print("   Check raw directory structure manually")
            return False
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Error downloading: {e}")
        print("\n💡 Try manual download:")
        print(f"   1. Go to: https://www.kaggle.com/datasets/{dataset_name}")
        print(f"   2. Click 'Download'")
        print(f"   3. Extract to: {CBIS_RAW_DIR}")
        return False

# Run setup (commented - uncomment to execute)
# setup_kaggle_automated()

print("""
📋 TO RUN AUTOMATED DOWNLOAD:

Uncomment last line and run cell:
# setup_kaggle_automated()

Or run manually:
setup_kaggle_automated()
""")


📋 TO RUN AUTOMATED DOWNLOAD:

Uncomment last line and run cell:
# setup_kaggle_automated()

Or run manually:
setup_kaggle_automated()



## 3. Check if Data is Downloaded

In [5]:
# Check for existing data
print("🔍 Checking for existing CBIS-DDSM data...")

# Check raw directory
raw_files = list(CBIS_RAW_DIR.glob('**/*'))
print(f"\nRaw directory: {len([f for f in raw_files if f.is_file()])} files")

# Check processed images
image_counts = {}
for split in ['train', 'val', 'test']:
    for label in ['benign', 'malignant']:
        path = CBIS_IMAGES_DIR / split / label
        count = len(list(path.glob('*.png')))
        image_counts[f'{split}/{label}'] = count

total_images = sum(image_counts.values())

print(f"\n📊 Processed images: {total_images} total")
for key, count in image_counts.items():
    print(f"  {key}: {count}")

if total_images == 0:
    print("""
⚠️ NO DATA FOUND

Please download CBIS-DDSM dataset first:

1. Install kaggle: pip install kaggle
2. Setup credentials: ~/.kaggle/kaggle.json
3. Download:
   cd data/cbis_ddsm/raw/
   kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset
   
Or manually download from:
https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset

Once downloaded, continue with this notebook for preprocessing.
""")
    DATA_AVAILABLE = False
else:
    print("\n✅ Data found! Ready for processing.")
    DATA_AVAILABLE = True

🔍 Checking for existing CBIS-DDSM data...

Raw directory: 0 files

📊 Processed images: 0 total
  train/benign: 0
  train/malignant: 0
  val/benign: 0
  val/malignant: 0
  test/benign: 0
  test/malignant: 0

⚠️ NO DATA FOUND

Please download CBIS-DDSM dataset first:

1. Install kaggle: pip install kaggle
2. Setup credentials: ~/.kaggle/kaggle.json
3. Download:
   cd data/cbis_ddsm/raw/
   kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset

Or manually download from:
https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset

Once downloaded, continue with this notebook for preprocessing.



## 4. DICOM to PNG Conversion (if needed)

### Function to convert medical DICOM format to standard PNG

In [6]:
def convert_dicom_to_png(dicom_path, png_path, target_size=(512, 512)):
    """
    Convert DICOM file to PNG image
    
    Args:
        dicom_path: Path to .dcm file
        png_path: Output path for .png file
        target_size: Resize to this size (width, height)
    
    Returns:
        True if successful, False otherwise
    """
    if not PYDICOM_AVAILABLE:
        print("⚠️ pydicom not installed. Install with: pip install pydicom")
        return False
    
    try:
        # Read DICOM
        dicom = pydicom.dcmread(dicom_path)
        
        # Get pixel array
        image = dicom.pixel_array
        
        # Normalize to 0-255
        image = ((image - image.min()) / (image.max() - image.min()) * 255).astype(np.uint8)
        
        # Resize
        image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
        
        # Save as PNG
        cv2.imwrite(str(png_path), image)
        
        return True
    
    except Exception as e:
        print(f"Error converting {dicom_path}: {e}")
        return False


def batch_convert_dicom_to_png(input_dir, output_dir, target_size=(512, 512), limit=None):
    """
    Batch convert all DICOM files in directory
    
    Args:
        input_dir: Directory with .dcm files
        output_dir: Output directory for .png files
        target_size: Resize images to this size
        limit: Maximum number of files to convert (for mini dataset)
    
    Returns:
        Number of files converted
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all DICOM files
    dicom_files = list(input_dir.rglob('*.dcm'))
    
    if limit:
        dicom_files = dicom_files[:limit]
    
    print(f"📁 Found {len(dicom_files)} DICOM files")
    print(f"🎯 Converting to PNG (target size: {target_size})...")
    
    converted = 0
    for i, dcm_path in enumerate(dicom_files):
        # Create output path
        png_name = dcm_path.stem + '.png'
        png_path = output_dir / png_name
        
        # Convert
        if convert_dicom_to_png(dcm_path, png_path, target_size):
            converted += 1
        
        # Progress
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(dicom_files)} files...")
    
    print(f"✅ Converted {converted} files successfully")
    return converted


# Example usage (commented - only run if you have DICOM files)
# converted_count = batch_convert_dicom_to_png(
#     input_dir=CBIS_RAW_DIR / 'Mass-Training_P_00001_LEFT_CC',
#     output_dir=CBIS_PROCESSED_DIR / 'temp',
#     target_size=(512, 512),
#     limit=100  # Limit for mini dataset
# )

print("✅ DICOM conversion functions defined")
print("   Use batch_convert_dicom_to_png() if you have .dcm files")

✅ DICOM conversion functions defined
   Use batch_convert_dicom_to_png() if you have .dcm files


## 5. Create Mini Dataset (Sample from Full)

### If full dataset is downloaded, create stratified mini subset

In [7]:
def create_mini_dataset(source_dir, output_dir, sample_ratio=0.3, target_size=(512, 512)):
    """
    Create mini dataset by sampling from full dataset
    
    Args:
        source_dir: Full dataset directory
        output_dir: Mini dataset output directory
        sample_ratio: Proportion to sample (0.3 = 30%)
        target_size: Resize images to this size
    
    Returns:
        Dictionary with statistics
    """
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)
    
    stats = {'benign': 0, 'malignant': 0}
    
    for label in ['benign', 'malignant']:
        # Find all images for this label
        image_files = list((source_dir / label).glob('*.png'))
        
        # Sample randomly
        num_samples = int(len(image_files) * sample_ratio)
        sampled_files = np.random.choice(image_files, size=num_samples, replace=False)
        
        # Create output directory
        (output_dir / label).mkdir(parents=True, exist_ok=True)
        
        # Copy and resize images
        for img_path in sampled_files:
            # Read image
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            
            # Resize
            img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
            
            # Save
            output_path = output_dir / label / img_path.name
            cv2.imwrite(str(output_path), img)
            
            stats[label] += 1
        
        print(f"✅ {label}: sampled {stats[label]} images from {len(image_files)}")
    
    return stats


# Example usage (commented)
# mini_stats = create_mini_dataset(
#     source_dir=CBIS_RAW_DIR / 'full_dataset',
#     output_dir=CBIS_PROCESSED_DIR / 'mini',
#     sample_ratio=0.25,  # 25% of full dataset
#     target_size=(512, 512)
# )

print("✅ Mini dataset creation function defined")
print("   Use create_mini_dataset() to sample from full dataset")

✅ Mini dataset creation function defined
   Use create_mini_dataset() to sample from full dataset


## 6. Train/Val/Test Split

In [8]:
def organize_train_val_test_split(source_dir, output_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Organize images into train/val/test splits
    
    Args:
        source_dir: Directory with benign/malignant subdirs
        output_dir: Output directory (will create train/val/test subdirs)
        train_ratio: Proportion for training (0.7 = 70%)
        val_ratio: Proportion for validation
        test_ratio: Proportion for test
    
    Returns:
        Dictionary with split statistics
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 0.01, "Ratios must sum to 1.0"
    
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)
    
    stats = {
        'train': {'benign': 0, 'malignant': 0},
        'val': {'benign': 0, 'malignant': 0},
        'test': {'benign': 0, 'malignant': 0}
    }
    
    for label in ['benign', 'malignant']:
        # Get all images
        image_files = list((source_dir / label).glob('*.png'))
        np.random.shuffle(image_files)
        
        # Calculate split indices
        n_total = len(image_files)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        
        # Split
        train_files = image_files[:n_train]
        val_files = image_files[n_train:n_train + n_val]
        test_files = image_files[n_train + n_val:]
        
        # Copy files
        for split, files in [('train', train_files), ('val', val_files), ('test', test_files)]:
            output_split_dir = output_dir / split / label
            output_split_dir.mkdir(parents=True, exist_ok=True)
            
            for img_path in files:
                # Read and save (use copy to preserve original)
                import shutil
                shutil.copy(img_path, output_split_dir / img_path.name)
                stats[split][label] += 1
        
        print(f"✅ {label}: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")
    
    return stats


# Example usage (commented)
# split_stats = organize_train_val_test_split(
#     source_dir=CBIS_PROCESSED_DIR / 'mini',
#     output_dir=CBIS_IMAGES_DIR,
#     train_ratio=0.7,
#     val_ratio=0.15,
#     test_ratio=0.15
# )

print("✅ Train/Val/Test split function defined")
print("   Use organize_train_val_test_split() to organize dataset")

✅ Train/Val/Test split function defined
   Use organize_train_val_test_split() to organize dataset


## 7. Image Preprocessing Functions

In [9]:
def preprocess_mammogram(image_path, target_size=(224, 224), enhance=True):
    """
    Preprocess mammogram image
    
    Args:
        image_path: Path to image file
        target_size: Resize to this size
        enhance: Apply contrast enhancement
    
    Returns:
        Preprocessed image as numpy array
    """
    # Read image
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    
    # Resize
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    
    if enhance:
        # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = clahe.apply(img)
    
    # Normalize to [0, 1]
    img = img.astype(np.float32) / 255.0
    
    return img


def augment_image(image, augmentation_params):
    """
    Apply data augmentation
    
    Args:
        image: Input image (numpy array)
        augmentation_params: Dict with augmentation settings
    
    Returns:
        Augmented image
    """
    # Random horizontal flip
    if augmentation_params.get('horizontal_flip', False):
        if np.random.rand() > 0.5:
            image = cv2.flip(image, 1)
    
    # Random rotation
    if 'rotation_range' in augmentation_params:
        angle = np.random.uniform(-augmentation_params['rotation_range'], 
                                 augmentation_params['rotation_range'])
        h, w = image.shape[:2]
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        image = cv2.warpAffine(image, M, (w, h))
    
    # Random zoom
    if 'zoom_range' in augmentation_params:
        zoom = np.random.uniform(1.0 - augmentation_params['zoom_range'], 
                                1.0 + augmentation_params['zoom_range'])
        h, w = image.shape[:2]
        new_h, new_w = int(h * zoom), int(w * zoom)
        image = cv2.resize(image, (new_w, new_h))
        
        # Crop/pad back to original size
        if zoom > 1.0:  # Crop
            start_h = (new_h - h) // 2
            start_w = (new_w - w) // 2
            image = image[start_h:start_h+h, start_w:start_w+w]
        else:  # Pad
            pad_h = (h - new_h) // 2
            pad_w = (w - new_w) // 2
            image = cv2.copyMakeBorder(image, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_CONSTANT, value=0)
            image = cv2.resize(image, (w, h))
    
    return image


# Define augmentation parameters for training
TRAIN_AUGMENTATION = {
    'horizontal_flip': True,
    'rotation_range': 10,  # degrees
    'zoom_range': 0.1,  # 10% zoom in/out
}

print("✅ Preprocessing functions defined:")
print("   - preprocess_mammogram(): CLAHE + normalize")
print("   - augment_image(): flip + rotate + zoom")
print(f"\n📋 Training augmentation params: {TRAIN_AUGMENTATION}")

✅ Preprocessing functions defined:
   - preprocess_mammogram(): CLAHE + normalize
   - augment_image(): flip + rotate + zoom

📋 Training augmentation params: {'horizontal_flip': True, 'rotation_range': 10, 'zoom_range': 0.1}


## 8. Visualize Sample Images (if available)

In [10]:
if DATA_AVAILABLE and total_images > 0:
    print_section_header("SAMPLE IMAGES VISUALIZATION")
    
    # Get sample images
    sample_images = []
    for label in ['benign', 'malignant']:
        train_dir = CBIS_IMAGES_DIR / 'train' / label
        images = list(train_dir.glob('*.png'))
        if len(images) > 0:
            # Take 3 samples
            samples = np.random.choice(images, size=min(3, len(images)), replace=False)
            for img_path in samples:
                sample_images.append((img_path, label))
    
    if len(sample_images) > 0:
        # Plot samples
        n_samples = len(sample_images)
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, (img_path, label) in enumerate(sample_images):
            if idx >= 6:
                break
            
            # Read image
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            
            # Plot
            axes[idx].imshow(img, cmap='gray')
            axes[idx].set_title(f'{label.upper()}\n{img_path.name}', fontsize=11, fontweight='bold')
            axes[idx].axis('off')
        
        # Hide unused subplots
        for idx in range(len(sample_images), 6):
            axes[idx].axis('off')
        
        plt.suptitle('Sample Mammogram Images', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        save_figure(fig, 'cbis_sample_images', RESULTS_DIR)
        print("✅ Sample images visualized")
    else:
        print("⚠️ No sample images found")
else:
    print("⚠️ Data not available - skipping visualization")
    print("   Download dataset first, then re-run this cell")

⚠️ Data not available - skipping visualization
   Download dataset first, then re-run this cell


## 9. Dataset Statistics

In [11]:
def calculate_dataset_statistics(images_dir):
    """
    Calculate comprehensive dataset statistics
    
    Args:
        images_dir: Root directory with train/val/test subdirs
    
    Returns:
        Dictionary with statistics
    """
    images_dir = Path(images_dir)
    
    stats = {
        'splits': {},
        'total': {'benign': 0, 'malignant': 0, 'all': 0}
    }
    
    for split in ['train', 'val', 'test']:
        stats['splits'][split] = {'benign': 0, 'malignant': 0}
        
        for label in ['benign', 'malignant']:
            path = images_dir / split / label
            count = len(list(path.glob('*.png')))
            stats['splits'][split][label] = count
            stats['total'][label] += count
            stats['total']['all'] += count
    
    return stats


if DATA_AVAILABLE and total_images > 0:
    print_section_header("DATASET STATISTICS")
    
    stats = calculate_dataset_statistics(CBIS_IMAGES_DIR)
    
    print("\n📊 DATASET SUMMARY:")
    print("="*60)
    
    for split in ['train', 'val', 'test']:
        split_stats = stats['splits'][split]
        total_split = split_stats['benign'] + split_stats['malignant']
        
        print(f"\n{split.upper()}:")
        print(f"  Benign:    {split_stats['benign']:4d} ({split_stats['benign']/total_split*100:.1f}%)")
        print(f"  Malignant: {split_stats['malignant']:4d} ({split_stats['malignant']/total_split*100:.1f}%)")
        print(f"  Total:     {total_split:4d}")
    
    print(f"\nTOTAL:")
    print(f"  Benign:    {stats['total']['benign']:4d} ({stats['total']['benign']/stats['total']['all']*100:.1f}%)")
    print(f"  Malignant: {stats['total']['malignant']:4d} ({stats['total']['malignant']/stats['total']['all']*100:.1f}%)")
    print(f"  All:       {stats['total']['all']:4d}")
    print("="*60)
    
    # Visualize distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Split distribution
    splits = ['train', 'val', 'test']
    benign_counts = [stats['splits'][s]['benign'] for s in splits]
    malignant_counts = [stats['splits'][s]['malignant'] for s in splits]
    
    x = np.arange(len(splits))
    width = 0.35
    
    axes[0].bar(x - width/2, benign_counts, width, label='Benign', color='#3498db', alpha=0.8)
    axes[0].bar(x + width/2, malignant_counts, width, label='Malignant', color='#e74c3c', alpha=0.8)
    axes[0].set_xlabel('Split', fontsize=12)
    axes[0].set_ylabel('Number of Images', fontsize=12)
    axes[0].set_title('Dataset Distribution by Split', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([s.upper() for s in splits])
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Class balance
    labels = ['Benign', 'Malignant']
    counts = [stats['total']['benign'], stats['total']['malignant']]
    colors = ['#3498db', '#e74c3c']
    
    axes[1].pie(counts, labels=labels, colors=colors, autopct='%1.1f%%', 
               startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
    axes[1].set_title('Overall Class Distribution', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    save_figure(fig, 'cbis_dataset_statistics', RESULTS_DIR)
    
    # Save statistics
    stats_df = pd.DataFrame({
        'Split': ['Train', 'Val', 'Test', 'Total'],
        'Benign': [stats['splits']['train']['benign'], 
                  stats['splits']['val']['benign'],
                  stats['splits']['test']['benign'],
                  stats['total']['benign']],
        'Malignant': [stats['splits']['train']['malignant'],
                     stats['splits']['val']['malignant'],
                     stats['splits']['test']['malignant'],
                     stats['total']['malignant']],
        'Total': [stats['splits']['train']['benign'] + stats['splits']['train']['malignant'],
                 stats['splits']['val']['benign'] + stats['splits']['val']['malignant'],
                 stats['splits']['test']['benign'] + stats['splits']['test']['malignant'],
                 stats['total']['all']]
    })
    
    stats_df.to_csv(RESULTS_DIR / 'cbis_dataset_stats.csv', index=False)
    print(f"\n💾 Statistics saved to: {RESULTS_DIR / 'cbis_dataset_stats.csv'}")
    
else:
    print("⚠️ Data not available - skipping statistics")
    print("\n📋 Expected dataset structure:")
    print("""
    data/cbis_ddsm/processed/images/
        train/
            benign/
            malignant/
        val/
            benign/
            malignant/
        test/
            benign/
            malignant/
    """)

⚠️ Data not available - skipping statistics

📋 Expected dataset structure:

    data/cbis_ddsm/processed/images/
        train/
            benign/
            malignant/
        val/
            benign/
            malignant/
        test/
            benign/
            malignant/
    


## 10. Summary & Next Steps

In [12]:
print_section_header("SUMMARY & NEXT STEPS")

print("""
✅ NOTEBOOK 08 COMPLETE

📋 WHAT WE'VE PREPARED:

1. ✅ Directory structure for CBIS-DDSM dataset
2. ✅ Download instructions (Kaggle API or manual)
3. ✅ DICOM to PNG conversion functions
4. ✅ Mini dataset sampling functions (5-10GB subset)
5. ✅ Train/Val/Test split organizer
6. ✅ Image preprocessing pipeline (CLAHE + normalization)
7. ✅ Data augmentation functions (flip, rotate, zoom)
8. ✅ Dataset statistics and visualization

📥 TO GET STARTED WITH DEEP LEARNING:

1. Download CBIS-DDSM dataset:
   - Kaggle: https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset
   - Or use download cell commands above
   
2. For MINI DATASET (5-10GB):
   - Use create_mini_dataset() with sample_ratio=0.25-0.30
   - Target: ~2000-3000 images total
   
3. Organize with organize_train_val_test_split()
   - 70% train, 15% val, 15% test
   
4. Dataset will be ready for CNN training in Notebook 09!

💡 ALTERNATIVE - SMALLER DATASETS:

If 5-10GB is still too large, consider:
- Mini-MIAS (smaller mammography dataset, ~300 images, <1GB)
- INbreast (Portuguese dataset, ~400 images, ~2GB)
- Or use synthetic data augmentation on even smaller sample

🎯 DATA REQUIREMENTS FOR RESEARCH:

Minimum viable: ~500-1000 images per class
Ideal: 2000-3000 images per class
Our mini target: 1000-1500 per class (~5GB total)

""")

if DATA_AVAILABLE and total_images > 0:
    print("✅ Data is ready! Proceed to Notebook 09 for CNN training.\n")
else:
    print("⚠️ Data not yet downloaded. Follow instructions above, then proceed.\n")

print("Next Notebook: 09_cbis_cnn_training.ipynb")
print("="*80)


                              SUMMARY & NEXT STEPS                              


✅ NOTEBOOK 08 COMPLETE

📋 WHAT WE'VE PREPARED:

1. ✅ Directory structure for CBIS-DDSM dataset
2. ✅ Download instructions (Kaggle API or manual)
3. ✅ DICOM to PNG conversion functions
4. ✅ Mini dataset sampling functions (5-10GB subset)
5. ✅ Train/Val/Test split organizer
6. ✅ Image preprocessing pipeline (CLAHE + normalization)
7. ✅ Data augmentation functions (flip, rotate, zoom)
8. ✅ Dataset statistics and visualization

📥 TO GET STARTED WITH DEEP LEARNING:

1. Download CBIS-DDSM dataset:
   - Kaggle: https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset
   - Or use download cell commands above

2. For MINI DATASET (5-10GB):
   - Use create_mini_dataset() with sample_ratio=0.25-0.30
   - Target: ~2000-3000 images total

3. Organize with organize_train_val_test_split()
   - 70% train, 15% val, 15% test

4. Dataset will be ready for CNN training in Notebook 09!

💡 ALTERNATIVE - 